In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

# Read Raw Data

In [2]:
df = pd.read_csv("first_25000_rows.csv")

df['ts_event'] = pd.to_datetime(df['ts_event'])

df = df.sort_values(by='ts_event').reset_index(drop=True)

# OFI Processing

In [3]:
def preprocess_OFI(df, level):
    # px: price, sz: size
    bid_px = df[f'bid_px_{level:02d}']
    bid_sz = df[f'bid_sz_{level:02d}']
    ask_px = df[f'ask_px_{level:02d}']
    ask_sz = df[f'ask_sz_{level:02d}']

    bid_px_prev = bid_px.shift(1)
    bid_sz_prev = bid_sz.shift(1)
    ask_px_prev = ask_px.shift(1)
    ask_sz_prev = ask_sz.shift(1)

    OFI_bid = np.where(
        bid_px > bid_px_prev, bid_sz,           # px_n > px_n-1
        np.where(bid_px < bid_px_prev, -bid_sz, # px_n < px_n-1
                 bid_sz - bid_sz_prev)          # px_n = px_n-1
    )

    OFI_ask = np.where(
        ask_px > ask_px_prev, -ask_sz,          # px_n > px_n-1
        np.where(ask_px < ask_px_prev, ask_sz,  # px_n < px_n-1
                 ask_sz - ask_sz_prev)          # px_n = px_n-1
    )
    
    total_size = np.array(bid_sz + ask_sz)
    
    return OFI_bid, OFI_ask, OFI_bid - OFI_ask, total_size

In [4]:
OFI_df       = pd.DataFrame()
OFI_df.index = df['ts_event'].copy()
OFI_df["symbol"] = np.array(df["symbol"].copy())
OFI_columns  = []

# Level from 1 - 10
for level in range(10):
    col_name1 = f'OFb_{level+1:02d}'
    col_name2 = f'OFa_{level+1:02d}'
    col_name3 = f'OFI_{level+1:02d}'
    col_name4 = f'total_size_{level+1:02d}'
    OFI_columns.append(col_name3)
    OFI_df[col_name1], OFI_df[col_name2], OFI_df[col_name3], OFI_df[col_name4] = preprocess_OFI(df, level)

In [5]:
# PCA Transformation

def calculate_PCA1_vec(data):
    matrix = data.fillna(0).values
    pca    = PCA(n_components=1)
    pca.fit(matrix)
    vec    = pca.components_ 
    return vec

PCA_W = {}
for s in df["symbol"].unique().tolist():
    PCA_W[s] = calculate_PCA1_vec(OFI_df[OFI_df["symbol"] == s][OFI_columns])

# Calculate 4 OFI

In [6]:
def calculate_4_OFI(OFI_df, PCA_W, start_time, end_time, symbol):
    
    observed_df = OFI_df[(OFI_df.index >= start_time) & (OFI_df.index <= end_time) ].copy().dropna()
    selected_df = observed_df[observed_df ["symbol"] == symbol].copy()
    
    # Best-Level OFI
    best  = selected_df["OFI_01"].sum()
    
    # Multi-Level OFI
    Q = 0
    for i in range(10):
        Q += selected_df[f'total_size_{i+1:02d}'].sum()
    Q /= (10 * 2 * len(selected_df))
    
    temp = []
    for i in range(10):
        col = f'OFI_{i+1:02d}'
        val = selected_df[col].sum() / Q
        temp.append(val)
    multi = np.array(temp).T
    
    # Integrated OFI
    W = PCA_W[symbol]
    integrated = np.dot(W, multi) /  np.linalg.norm(W, ord=1)
    
    return best, multi, integrated[0]

$\textbf{Cross-Asset OFI}$
 
We define as the sum of all Integrated OFIs.

In [7]:
# Input Variables

start_time, end_time = "2024-10-21 11:55:00", "2024-10-21 13:00:00"
symbol = "AAPL"

In [8]:
OFIs = {}

for s in df["symbol"].unique().tolist():
    OFIs[s] = calculate_4_OFI(OFI_df, PCA_W, start_time, end_time, s)

Cross_Asset_OFI = sum([OFIs[s][-1] for x in df["symbol"].unique().tolist()])

In [9]:
Best_Level_OFI, Multi_Level_OFI, Integrated_OFI = OFIs[symbol]

In [10]:
print(Best_Level_OFI, Multi_Level_OFI, Integrated_OFI, Cross_Asset_OFI)

-65571.0 [-323.03472896 -265.85298552 -119.17173893  544.74638414   -3.3401587
  321.88685731  541.80527094   98.51004926  -94.59841798   19.17389036] 789.1191814882321 789.1191814882321
